In [3]:
import pandas as pd

# Load the dataset (already done)
df = pd.read_csv("spam.csv", encoding='ISO-8859-1')

# Rename and keep only relevant columns
df = df.rename(columns={'v1': 'label', 'v2': 'text'})
df = df[['label', 'text']]

# Convert labels to binary: ham → 0, spam → 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df.head()


,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
import re
import string
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove digits
    text = text.translate(str.maketrans("", "", string.punctuation))  # Remove punctuation
    words = text.split()
    words = [word for word in words if word not in stopwords.words('english')]
    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['label'], test_size=0.2, random_state=42)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Predict and evaluate
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred))


Accuracy: 0.9722
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       965
           1       1.00      0.79      0.88       150

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [7]:
import joblib

joblib.dump(model, "spam_classifier.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']

In [8]:
# Your sample email
new_email = ["You have een selected to win a lottery!"]

# Preprocess and vectorize
cleaned = preprocess_text(new_email[0])
email_vector = vectorizer.transform([cleaned])

# Predict
prediction = model.predict(email_vector)[0]
probability = model.predict_proba(email_vector)[0][1] * 100

print("Prediction:", "Spam" if prediction == 1 else "Not Spam")
print(f"Spam Probability: {probability:.2f}%")


Prediction: Spam
Spam Probability: 64.91%


In [9]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer  # Or other summarizer (e.g., LexRank)

In [10]:
import nltk
nltk.data.path = ['./nltk_data'] + nltk.data.path
print("NLTK data paths:")
for path in nltk.data.path:
    print(f"- '{path}'")
nltk.download('punkt', download_dir='./nltk_data')  # Ensure punkt is here

NLTK data paths:
- './nltk_data'
- 'C:\Users\hp/nltk_data'
- 'C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\nltk_data'
- 'C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\share\nltk_data'
- 'C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\lib\nltk_data'
- 'C:\Users\hp\AppData\Roaming\nltk_data'
- 'C:\nltk_data'
- 'D:\nltk_data'
- 'E:\nltk_data'
- 'C:\Users\hp\OneDrive\Desktop\spam\nltk_data\tokenizers\punkt'


[nltk_data] Downloading package punkt to ./nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [11]:
def summarize_text(text, sentences_count=2):  # You can adjust sentences_count
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = LsaSummarizer()  # Or LexRankSummarizer()
    summary = summarizer(parser.document, sentences_count)
    return " ".join(str(sentence) for sentence in summary)

In [12]:
import nltk

# Using double backslashes to avoid escape characters
nltk.data.path.append('C:\\Users\\hp\\OneDrive\\Desktop\\spam\\nltk_data\\tokenizers\\punkt')

# Your sample email (from your notebook)
new_email = ["You have been selected to win a $1000 Amazon gift card! This is a limited-time offer. Click here to claim your prize! Don't miss out on this amazing opportunity!"]

# Preprocess and vectorize (as you already do)
cleaned = preprocess_text(new_email[0])
email_vector = vectorizer.transform([cleaned])

# Predict (as you already do)
prediction = model.predict(email_vector)[0]
probability = model.predict_proba(email_vector)[0][1] * 100

print("Prediction:", "Spam" if prediction == 1 else "Not Spam")
print(f"Spam Probability: {probability:.2f}%")

# Summarize the email
summary = summarize_text(new_email[0], sentences_count=2)
print("\nSummary:")
print(summary)

Prediction: Spam
Spam Probability: 83.20%


LookupError: NLTK tokenizers are missing or the language is not supported.
Download them by following command: python -c "import nltk; nltk.download('punkt')"
Original error was:

**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - './nltk_data'
    - 'C:\\Users\\hp/nltk_data'
    - 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\nltk_data'
    - 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\share\\nltk_data'
    - 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\lib\\nltk_data'
    - 'C:\\Users\\hp\\AppData\\

In [ ]:
import nltk
nltk.download()


showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml
